In [5]:
import geopandas as gpd
import rasterio
from rasterio.features import rasterize
from rasterio.transform import from_origin
import numpy as np

def polygon_to_raster_by_field(shapefile_path, out_raster_path, field_name, resolution=10):
    # 读取 shapefile
    gdf = gpd.read_file(shapefile_path)

    # 提取边界信息
    minx, miny, maxx, maxy = gdf.total_bounds
    width = int((maxx - minx) / resolution)
    height = int((maxy - miny) / resolution)

    # 构造仿射变换矩阵
    transform = from_origin(minx, maxy, resolution, resolution)

    # 构造 rasterize 输入 [(geometry, value), ...]
    shapes = [
        (geom, value)
        for geom, value in zip(gdf.geometry, gdf[field_name])
        if geom is not None and not geom.is_empty
    ]

    # 栅格化
    raster = rasterize(
        shapes,
        out_shape=(height, width),
        transform=transform,
        fill=0,                # 背景值
        dtype="int32"          # 输出值的数据类型
    )

    # 写入为 GeoTIFF
    with rasterio.open(
        out_raster_path,
        "w",
        driver="GTiff",
        height=height,
        width=width,
        count=1,
        dtype=raster.dtype,
        crs=gdf.crs,
        transform=transform
    ) as dst:
        dst.write(raster, 1)

# 用法示例
polygon_to_raster_by_field(
    shapefile_path="D:/shanghai workshop/vector/landuse_park.shp",
    out_raster_path="D:/shanghai workshop/raster/park.tif",
    field_name="FID",      # 或其他字段，比如 'type'、'id'、'code'
    resolution=0.15        # 分辨率：单位为 shapefile 的坐标单位（一般是米）
)


ImportError: DLL load failed while importing _multiarray_umath: The specified module could not be found.

ImportError: numpy.core.multiarray failed to import